# GrowWithMe — Offline Nana NLU **v2**

Bigger, smarter successor to v1 (which trained on 408 template rows):
- **Thousands of generated examples** from a slot-filling engine grounded in how people
  actually talk in Ghana: Pidgin constructions ("e dey hot oo", "she get belle",
  "e no dey chop"), SMS-style typos ("pls", "wat"), local foods (TZ, banku, koko,
  weanimix, ayoyo), and "weighing"/ANC clinic vocabulary.
- **4 new intents**: ask_danger_signs, ask_feeding_question, ask_vaccine, find_clinic —
  the app answers each with curated GHS-based text (never generated).
- **Hard negatives** so off-topic messages land in help_other instead of a wrong action.
- **A hand-written hard test** the generator never saw — the honest accuracy number.

**Run top to bottom on Colab (CPU fine, ~10–15 min).** Distillation cell is optional.

## Featurizer contract (MUST match `mobile/lib/data/model/nlu_service.dart` — UNCHANGED from v1)
lowercase; `[^a-z0-9' ]`→space; word unigrams `u:`, bigrams `b:_`, char trigrams `c:` of
`^tok$`; FNV-1a 32-bit % **8192**; L2 norm. Output = `[intent probs] + [subject probs]`.

In [ ]:
%pip -q install tensorflow scikit-learn numpy requests
import hashlib, json, random, re
import numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
random.seed(42); np.random.seed(42); tf.random.set_seed(42)

INTENTS = ['start_health_check', 'open_add_child', 'open_add_pregnancy', 'plan_diet',
           'read_today', 'get_tip', 'log_weight', 'set_reminder', 'greeting', 'help_other',
           'ask_danger_signs', 'ask_feeding_question', 'ask_vaccine', 'find_clinic']
SUBJECTS = ['child', 'pregnancy', 'unknown']
BUCKETS = 8192

def fnv1a32(s):
    h = 2166136261
    for b in s.encode('utf-8'):
        h ^= b; h = (h * 16777619) & 0xFFFFFFFF
    return h

def featurize(text):
    t = re.sub(r'[^a-z0-9\' ]', ' ', text.lower())
    toks = [w for w in t.split(' ') if w]
    feats = []
    for w in toks:
        feats.append('u:' + w)
        p = '^' + w + '$'
        feats += ['c:' + p[i:i+3] for i in range(len(p) - 2)]
    feats += ['b:' + a + '_' + b for a, b in zip(toks, toks[1:])]
    v = np.zeros(BUCKETS, dtype='float32')
    for f in feats:
        v[fnv1a32(f) % BUCKETS] += 1
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

# Contract check — must keep matching mobile/test/nlu_featurizer_test.dart:
probe = featurize('my baby has fever')
assert fnv1a32('u:fever') % 8192 == 4112 and int((probe > 0).sum()) == 21
print('featurizer contract OK')

## 1. The generation engine — Ghana-grounded slots + noise

In [ ]:
CHILD_REF = ['my baby', 'my child', 'my son', 'my daughter', 'the baby', 'my small girl',
  'my small boy', 'the child', 'my little one', 'my second born', 'my last born', 'e']
CHILD_SYMPTOM = ['fever', 'hot body', 'body dey hot', 'convulsions', 'fits', 'shaking',
  'jerking', 'vomiting everything', 'dey vomit anyhow', 'diarrhoea', 'running stomach',
  'toilet plenty', 'watery stool', 'blood in the stool', 'coughing bad', 'catarrh and cough',
  'fast breathing', 'breathing hard', 'chest dey pull in', 'not eating', 'no dey chop',
  'refusing breast', "won't take breast", 'not sucking', 'very weak', 'weak like rag',
  'sleeping too much', 'no dey wake up well', 'yellow eyes', 'rash all over the body',
  'crying nonstop', 'crying since morning', 'sunken eyes', 'dry mouth', 'stomach swollen']
PREG_SYMPTOM = ['bleeding', 'spotting blood', 'blood dey come', 'severe headache',
  'my head dey pain me bad', 'blurred vision', 'seeing double', 'swollen feet',
  'my face and hands swell', 'baby not moving', 'baby no dey move', 'baby stopped kicking',
  'strong belly pain', 'my belly dey pain me', 'water broke early', 'water don break',
  'fever and chills', 'hot body with shivering', 'too weak to stand', 'dizzy anytime I stand',
  'vomiting everything I eat', 'waist pain bad']
TIME_TAIL = ['', ' since morning', ' since yesterday', ' since last night', ' for two days now',
  ' since three days', ' small small since last week', ' oo', ' right now']
ASK_TAIL = ['', ' what should I do', ' what do I do', ' abeg help me', ' please help',
  ' I beg you', ' wetin I go do', ' help me nana']
PREFIX = ['', 'please ', 'abeg ', 'nana ', 'madam ', 'good morning nana ', 'sorry oo ']
FOODS = ['TZ', 'tuo zaafi', 'banku', 'kenkey', 'fufu', 'koko', 'porridge', 'weanimix',
  'gari', 'beans', 'groundnut soup', 'ayoyo soup', 'alefu', 'dawadawa', 'okro soup',
  'rice', 'yam', 'maize', 'eggs', 'small fish', 'soya beans']

def sms_noise(s):
    """SMS-style typos: word shortenings + occasional dropped letter."""
    subs = {'please': 'pls', 'what': 'wat', 'should': 'shud', 'you': 'u', 'your': 'ur',
            'good': 'gud', 'morning': 'morn', 'because': 'cos', 'and': 'nd', 'the': 'de'}
    words = s.split(' ')
    out = []
    for w in words:
        lw = w.lower()
        if lw in subs and random.random() < 0.5:
            out.append(subs[lw])
        elif len(w) > 4 and random.random() < 0.12:
            i = random.randrange(1, len(w) - 1)
            out.append(w[:i] + w[i+1:])
        else:
            out.append(w)
    return ' '.join(out)

def emit(data, text, intent, subject, noisy=True):
    data.append((text, intent, subject))
    if noisy and random.random() < 0.45:
        data.append((sms_noise(text), intent, subject))

R = random.choice

In [ ]:
data = []

# --- start_health_check (child): ~1200 ---
for _ in range(600):
    c, s = R(CHILD_REF), R(CHILD_SYMPTOM)
    t = R([f'{c} has {s}', f'{c} get {s}', f'{c} is {s}', f'{c} dey {s}',
           f'{c} has {s}{R(TIME_TAIL)}', f'{c} {s}{R(TIME_TAIL)}{R(ASK_TAIL)}'])
    emit(data, R(PREFIX) + t, 'start_health_check', 'child')

# --- start_health_check (pregnancy): ~900 ---
for _ in range(450):
    s = R(PREG_SYMPTOM)
    t = R([f'I have {s}', f'I get {s}', f'I dey feel {s}', f'{s}{R(TIME_TAIL)}{R(ASK_TAIL)}',
           f'I am pregnant and I have {s}', f'me I get belle and {s}{R(TIME_TAIL)}'])
    emit(data, R(PREFIX) + t, 'start_health_check', 'pregnancy')

# --- start_health_check (unknown subject): ~200 ---
for t in ['I am not feeling well', 'I no dey feel fine', 'somebody dey sick for house',
          'I feel sick', 'we are not well today', 'something is wrong with me',
          'I need to check my health', 'run the health check', 'start the checker',
          'check me', 'do the sickness check', 'my body dey pain me', 'I dey feel some way'] * 8:
    emit(data, R(PREFIX) + t + R(ASK_TAIL), 'start_health_check', 'unknown')

# --- open_add_child: ~350 ---
for t in ['add my child', 'register my baby', 'I want to add my new baby',
          'I just delivered last week', 'I gave birth yesterday', 'I born yesterday',
          'I don born', 'my baby came on monday', 'put my daughter inside the app',
          'new baby for house', 'I delivered a baby boy', 'add another child',
          'register my son', 'how do I add my baby', 'I want to track my child',
          'my sister brought her baby to stay with me can I add him',
          'we welcomed a baby girl', 'God gave us a boy last night'] * 10:
    emit(data, R(PREFIX) + t, 'open_add_child', 'child')

# --- open_add_pregnancy: ~350 ---
for t in ['I am pregnant', 'I think I am pregnant', 'track my pregnancy',
          'I missed my period two months now', 'my period no come since',
          'start pregnancy tracking', 'I am expecting', 'me I get belle',
          'I dey carry belle', 'add my pregnancy', 'I am three months pregnant',
          'I want to follow my pregnancy', 'new pregnancy', 'the test came positive',
          'nurse said I am pregnant', 'I am with child'] * 11:
    emit(data, R(PREFIX) + t, 'open_add_pregnancy', 'pregnancy')

# --- plan_diet: ~700 with local foods ---
for _ in range(350):
    f1, f2 = R(FOODS), R(FOODS)
    t = R(['what should I cook today', 'plan my meals', 'what can we chop today',
           f'I only have {f1} and {f2} at home what can I make',
           f'is {f1} good for my baby', f'can I give {f1} to the child',
           'I have small money what can I cook', 'chop money no dey what do we eat',
           'help me with food', 'wetin I go cook', 'what do I feed her',
           'meal plan please', 'what soup is good for pregnancy',
           f'how do I prepare {f1} for the baby', 'plan food for the week',
           'what should a pregnant woman eat', 'my money no reach for fish',
           'give me one day meal plan', 'food for my baby'])
    emit(data, R(PREFIX) + t, 'plan_diet', 'unknown')

# --- read_today: ~300 ---
for t in ['what is happening today', 'do I have a visit today', 'when is my next clinic day',
          'read my reminders', 'what do I have this week', 'any appointment coming',
          'when do I go to the clinic again', 'tell me my schedule', 'weighing day is when',
          'when is the next weighing', 'when be my ANC', 'my antenatal is when',
          'anything for me today', 'what dey come next'] * 11:
    emit(data, R(PREFIX) + t, 'read_today', 'unknown')

# --- get_tip: ~250 ---
for t in ['give me a tip', 'feeding advice please', 'any advice for today',
          'teach me something', 'what should I know today', 'tips for feeding my baby',
          'advice for me', 'teach me small thing today', 'todays tip',
          'how do I keep my baby healthy'] * 13:
    emit(data, R(PREFIX) + t, 'get_tip', 'unknown')

# --- log_weight: ~250 ---
for _ in range(125):
    kg = round(random.uniform(2.5, 16.0), 1)
    t = R(['save my baby weight', 'record the weight from weighing',
           f'my child weighs {kg} kilos now', f'they weighed her today {kg}',
           'log weight', 'update his weight', f'the nurse said {kg} kg today',
           f'weighing came to {kg}', 'write down the weight from the card'])
    emit(data, R(PREFIX) + t, 'log_weight', 'child')

# --- set_reminder: ~250 ---
for t in ['remind me to go to the clinic', 'set a reminder for friday',
          'dont let me forget my appointment', 'remind me tomorrow morning',
          'alarm me for the weighing day', 'put a reminder for next tuesday',
          'remind me to take my medicine', 'wake me up for ANC day',
          'no let me forget the clinic oo'] * 14:
    emit(data, R(PREFIX) + t, 'set_reminder', 'unknown')

# --- greeting: ~250 ---
for t in ['hello', 'hi nana', 'good morning', 'good evening nana', 'how are you',
          'how you dey', 'you dey', 'thank you', 'thank you nana', 'God bless you',
          'you have helped me', 'ok', 'yes please', 'goodnight', 'medaase',
          'how are you doing today', 'wassup nana', 'ete sen'] * 8:
    emit(data, t, 'greeting', 'unknown')

# --- help_other + HARD NEGATIVES: ~400 ---
for t in ['what can you do', 'help', 'how does this app work', 'I need help with the app',
          'what is this', 'teach me how to use this', 'where do I see my children',
          'can I use this without internet', 'change my language', 'who made this app',
          'delete my account', 'my phone is misbehaving', 'the app is slow',
          # hard negatives — off-topic must NOT trigger actions
          'send money to my brother', 'what time is the football match',
          'I want to sell my goat', 'how much is fuel now', 'play me a song',
          'what is the weather tomorrow', 'call my husband', 'charge my phone',
          'the election results', 'fix my radio', 'I lost my wallet',
          'my neighbour is annoying me', 'tell me a story', 'what is 2 plus 2'] * 8:
    emit(data, t, 'help_other', 'unknown')

# --- ask_danger_signs (NEW): ~300 ---
for t in ['what are the danger signs', 'danger signs in pregnancy',
          'when should I rush to hospital', 'what signs mean my baby is in danger',
          'how do I know say the sickness is serious', 'which symptoms are dangerous',
          'what should make me go to clinic quick', 'signs that pregnancy has problem',
          'how will I know if my child is very sick', 'wetin be the danger signs',
          'teach me the warning signs', 'what danger signs should I watch for'] * 12:
    emit(data, R(PREFIX) + t, 'ask_danger_signs', 'unknown')

# --- ask_feeding_question (NEW): ~350 ---
for t in ['how often should I breastfeed', 'when do I start giving water',
          'at what age does baby eat solid food', 'is breast milk alone enough',
          'how many times should the baby chop a day', 'can I stop breastfeeding now',
          'when do I stop breast', 'what age for porridge', 'my milk no dey flow well',
          'baby is six months what food do I start', 'how do I wean the baby',
          'can baby take egg', 'when can baby eat fish', 'breastfeeding at night necessary',
          'how long exclusive breastfeeding'] * 10:
    emit(data, R(PREFIX) + t, 'ask_feeding_question', 'child')

# --- ask_vaccine (NEW): ~300 ---
for t in ['when is the next vaccine', 'which injections does my baby need',
          'what is BCG for', 'when do they give measles injection',
          'my baby missed a vaccine what do I do', 'is weighing the same as injection day',
          'vaccine schedule for babies', 'what immunization at six weeks',
          'do vaccines have side effects', 'the injection place is swollen',
          'why so many injections for small baby', 'when be the next weighing injection'] * 11:
    emit(data, R(PREFIX) + t, 'ask_vaccine', 'child')

# --- find_clinic (NEW): ~250 ---
for t in ['where is the nearest clinic', 'which hospital is close to me',
          'where can I go for help now', 'find me a health center',
          'is there a CHPS compound near here', 'where do I go for delivery',
          'which facility should I visit', 'hospital near my village',
          'where is the closest midwife', 'clinic for this area'] * 11:
    emit(data, R(PREFIX) + t, 'find_clinic', 'unknown')

random.shuffle(data)
print('generated examples:', len(data))
from collections import Counter
print(Counter(i for _, i, _ in data))

## 2. Optional: distill extra diversity from the online Nana LLM
More batches than v1, targeted at one intent per request so thin classes get depth.

In [ ]:
NVIDIA_API_KEY = ''  # <-- backend .env NVIDIA_API_KEY (optional)
NVIDIA_MODEL = 'meta/llama-3.3-70b-instruct'

if NVIDIA_API_KEY:
    import requests
    added = 0
    for intent in INTENTS:
        for batch in range(3):
            prompt = (
                'You create training data for a maternal-health app used by mothers around '
                'Tamale, Northern Ghana. Generate 30 short, realistic, VARIED things a '
                f'caregiver might type or say that mean the intent "{intent}". Use simple '
                'Ghanaian English with occasional Pidgin (dey, abeg, belle, chop, oo), local '
                'foods (TZ, banku, koko, weanimix), small typos. Subject is child, pregnancy '
                'or unknown. Output ONLY JSON lines: {"text": ..., "subject": ...}\n'
                f'Batch {batch}.')
            try:
                r = requests.post('https://integrate.api.nvidia.com/v1/chat/completions',
                    headers={'Authorization': f'Bearer {NVIDIA_API_KEY}'},
                    json={'model': NVIDIA_MODEL, 'temperature': 1.0,
                          'messages': [{'role': 'user', 'content': prompt}]},
                    timeout=120)
                for line in r.json()['choices'][0]['message']['content'].splitlines():
                    line = line.strip().strip('`')
                    if not line.startswith('{'): continue
                    try:
                        ex = json.loads(line)
                        if ex.get('subject') in SUBJECTS and ex.get('text'):
                            data.append((ex['text'], intent, ex['subject'])); added += 1
                    except Exception: pass
            except Exception as e:
                print('batch failed:', intent, e)
    print('LLM-distilled examples added:', added)
else:
    print('No API key — training on generated templates only.')

## 3. Train (Dense-128, label smoothing via dropout)

In [ ]:
random.shuffle(data)
X = np.stack([featurize(t) for t, _, _ in data])
yi = np.array([INTENTS.index(i) for _, i, _ in data])
ys = np.array([SUBJECTS.index(s) for _, _, s in data])
Xtr, Xte, yitr, yite, ystr, yste = train_test_split(X, yi, ys, test_size=0.12,
                                                    stratify=yi, random_state=42)

inp = tf.keras.Input(shape=(BUCKETS,))
h = tf.keras.layers.Dense(128, activation='relu')(inp)
h = tf.keras.layers.Dropout(0.25)(h)
intent_out = tf.keras.layers.Dense(len(INTENTS), activation='softmax', name='intent')(h)
subject_out = tf.keras.layers.Dense(len(SUBJECTS), activation='softmax', name='subject')(h)
combined = tf.keras.layers.Concatenate(name='combined')([intent_out, subject_out])
model = tf.keras.Model(inp, combined)

def split_loss(y_true, y_pred):
    yi_t, ys_t = y_true[:, 0], y_true[:, 1]
    pi, ps = y_pred[:, :len(INTENTS)], y_pred[:, len(INTENTS):]
    return (tf.keras.losses.sparse_categorical_crossentropy(yi_t, pi)
            + 0.4 * tf.keras.losses.sparse_categorical_crossentropy(ys_t, ps))

model.compile(optimizer=tf.keras.optimizers.Adam(2e-3), loss=split_loss)
y_tr = np.stack([yitr, ystr], axis=1).astype('float32')
y_te = np.stack([yite, yste], axis=1).astype('float32')
model.fit(Xtr, y_tr, validation_data=(Xte, y_te), epochs=30, batch_size=128, verbose=0,
          callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])

pred = model.predict(Xte, verbose=0)
print('=== INTENT (held-out split of generated data) ===')
print(classification_report(yite, pred[:, :len(INTENTS)].argmax(1), target_names=INTENTS, digits=3))
print('=== SUBJECT ===')
print(classification_report(yste, pred[:, len(INTENTS):].argmax(1), target_names=SUBJECTS, digits=3))

## 4. The HARD test — hand-written, never seen by the generator
This is the number that matters. Ship if hard-test intent accuracy ≥ 0.85 and no
symptom sentence lands outside start_health_check.

In [ ]:
HARD = [
    ('de pikin body dey hot en no chop since yesterday', 'start_health_check', 'child'),
    ('blood come when I go toilet and I get belle', 'start_health_check', 'pregnancy'),
    ('my wife belly dey pain her bad bad', 'start_health_check', 'pregnancy'),
    ('small girl dey stool water plenty times', 'start_health_check', 'child'),
    ('I born new baby for hospital yesterday evening', 'open_add_child', 'child'),
    ('menses no show for 8 weeks I think say I carry', 'open_add_pregnancy', 'pregnancy'),
    ('only gari and small beans dey house wetin we go chop', 'plan_diet', 'unknown'),
    ('abeg which day be my next ANC', 'read_today', 'unknown'),
    ('nurse weighed am today e be 7 point 5', 'log_weight', 'child'),
    ('no make I forget Thursday clinic oo', 'set_reminder', 'unknown'),
    ('maakye nana', 'greeting', 'unknown'),
    ('how I go take use dis app sef', 'help_other', 'unknown'),
    ('wetin go show say the belle get problem', 'ask_danger_signs', 'unknown'),
    ('baby don reach 6 months which food I go start am', 'ask_feeding_question', 'child'),
    ('dem say measles injection be when', 'ask_vaccine', 'child'),
    ('which hospital dey near our village for delivery', 'find_clinic', 'unknown'),
    ('buy me credit for my phone', 'help_other', 'unknown'),
    ('the baby fall from the bed and now e dey sleep too much', 'start_health_check', 'child'),
    ('is banku with okro ok for a 1 year old', 'plan_diet', 'unknown'),
    ('I wan know the warning signs for small babies', 'ask_danger_signs', 'unknown'),
]
ok_i = ok_s = 0
for text, want_i, want_s in HARD:
    out = model.predict(featurize(text).reshape(1, -1), verbose=0)[0]
    gi = INTENTS[out[:len(INTENTS)].argmax()]; gs = SUBJECTS[out[len(INTENTS):].argmax()]
    mark = '✅' if gi == want_i else '❌'
    ok_i += gi == want_i; ok_s += gs == want_s
    print(f'{mark} {text!r:55s} -> {gi:22s} (want {want_i}) subj={gs}')
print(f'\nHARD intent accuracy: {ok_i}/{len(HARD)}   subject: {ok_s}/{len(HARD)}')

## 5. Export TFLite v2 + manifest

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
tfl = conv.convert()
open('nana_nlu.tflite', 'wb').write(tfl)
sha = hashlib.sha256(tfl).hexdigest()
manifest = {
    'name': 'nana-nlu', 'version': 2, 'kind': 'tflite',
    'sizeBytes': len(tfl), 'sha256': sha,
    'intents': INTENTS, 'subjects': SUBJECTS, 'buckets': BUCKETS,
    'featurizer': 'lowercase; [^a-z0-9\\x27 ]->space; unigrams u:, bigrams b:_, char trigrams c: of ^tok$; fnv1a32 % buckets; L2 norm',
    'outputLayout': 'concat: intents then subjects',
    'minConfidence': 0.5,
    'trainedOn': f'{len(data)} examples (Ghana-grounded generator + LLM distillation), v2',
    'disclaimer': 'Understanding only — replies are curated in-app; model never generates text.',
}
json.dump(manifest, open('manifest.json', 'w'), indent=2)
print(f'nana_nlu.tflite: {len(tfl)/1024:.0f} KB\nsha256: {sha}')

In [ ]:
import requests
CLOUD_NAME = ''     # <-- your Cloudinary cloud name
UPLOAD_PRESET = ''  # <-- unsigned upload preset
up = requests.post(
    f'https://api.cloudinary.com/v1_1/{CLOUD_NAME}/raw/upload',
    files={'file': ('nana_nlu_v2.tflite', tfl)},
    data={'upload_preset': UPLOAD_PRESET, 'public_id': 'growwithme/models/nana_nlu_v2'},
).json()
print(up.get('secure_url') or up)
manifest['url'] = up['secure_url']
json.dump(manifest, open('manifest.json', 'w'), indent=2)
print(json.dumps(manifest, indent=2))